# Atividade 1 - Aprendizado de Máquina
### Perceptron Treinável, KNN e Recomendação por Similaridade

Este notebook está dividido em três seções, uma para cada desafio:

1. **Desafio 1** — Perceptron Treinável (detecção de fraude)
2. **Desafio 2** — KNN (predição de risco de churn)
3. **Desafio 3** — Recomendação de servidores por similaridade espacial

Todas as seções compartilham o mesmo `import numpy as np`, executado uma única vez na célula abaixo.

In [1]:
import numpy as np

---
## Desafio 1: Classificação Binária com Perceptron Treinável

**Contexto:** detectar transações financeiras suspeitas com base em duas métricas:
- `x1`: Valor da Transação Normalizado (0.0 a 10.0)
- `x2`: Frequência de Operações Recentes (últimos 5 minutos)

O modelo é treinado com a **regra de aprendizado de Rosenblatt**, que ajusta pesos e viés a cada erro de predição, até convergir (ou atingir o número máximo de épocas).

### 1.1 Funções do Perceptron
Função de ativação, função de treinamento (Rosenblatt) e função de inferência.

In [2]:
def funcao_degrau(z):
    """Função de ativação degrau (step function)."""
    return 1 if z >= 0 else 0


def treinar_perceptron(X, y, taxa_aprendizado=0.1, epocas=20):
    """
    Treina um Perceptron de Rosenblatt.
    Inicializa pesos e viés com 1.0 e ajusta a cada amostra com erro de predição.
    Retorna os pesos (w) e o viés (b) calibrados.
    """
    n_amostras, n_features = X.shape
    pesos = np.ones(n_features)
    vies = 1.0

    for epoca in range(1, epocas + 1):
        erros_na_epoca = 0
        for i in range(n_amostras):
            x_i = X[i]
            y_real = y[i]

            z = np.dot(x_i, pesos) + vies
            y_previsto = funcao_degrau(z)
            erro = y_real - y_previsto

            if erro != 0:
                pesos = pesos + taxa_aprendizado * erro * x_i
                vies = vies + taxa_aprendizado * erro
                erros_na_epoca += 1

        print(f"Epoca {epoca:2d} | Erros: {erros_na_epoca} | "
              f"Pesos: {np.round(pesos, 4)} | Vies: {round(vies, 4)}")

        if erros_na_epoca == 0:
            print(f"-> Convergencia atingida na epoca {epoca}.\n")
            break

    return pesos, vies


def prever(amostra, pesos, vies):
    """Calcula a combinação linear e aplica a função degrau."""
    z = np.dot(amostra, pesos) + vies
    return funcao_degrau(z)

### 1.2 Dados de Treino e Amostras de Teste

In [3]:
X_treino_fraude = np.array([
    [1.5, 1.0],
    [2.0, 2.0],
    [3.5, 1.5],
    [3.0, 3.0],
    [6.5, 5.0],
    [7.0, 7.0],
    [8.5, 6.0],
    [9.0, 8.0]
])

y_treino_fraude = np.array([0, 0, 0, 0, 1, 1, 1, 1])

transacao_A = np.array([5.0, 4.0])
transacao_B = np.array([7.0, 6.5])

### 1.3 Execução e Diagnóstico

In [4]:
pesos_finais, vies_final = treinar_perceptron(
    X_treino_fraude, y_treino_fraude, taxa_aprendizado=0.1, epocas=20
)

print("=== Parâmetros Calibrados ===")
print(f"Pesos finais (w1, w2): {pesos_finais}")
print(f"Viés final (b): {vies_final}\n")

status = {0: "LEGÍTIMA", 1: "SUSPEITA (Risco de Fraude)"}

pred_A = prever(transacao_A, pesos_finais, vies_final)
pred_B = prever(transacao_B, pesos_finais, vies_final)

print("=== Diagnóstico das Novas Transações ===")
print(f"Transação A {list(transacao_A)} -> Classe: {pred_A} -> {status[pred_A]}")
print(f"Transação B {list(transacao_B)} -> Classe: {pred_B} -> {status[pred_B]}")

Epoca  1 | Erros: 4 | Pesos: [-0.    0.25] | Vies: 0.6
Epoca  2 | Erros: 3 | Pesos: [0.3  0.45] | Vies: 0.5
Epoca  3 | Erros: 4 | Pesos: [0.25 0.5 ] | Vies: 0.3
Epoca  4 | Erros: 4 | Pesos: [0.2  0.55] | Vies: 0.1
Epoca  5 | Erros: 4 | Pesos: [0.2  0.45] | Vies: -0.1
Epoca  6 | Erros: 3 | Pesos: [0.5  0.65] | Vies: -0.2
Epoca  7 | Erros: 4 | Pesos: [0.45 0.7 ] | Vies: -0.4
Epoca  8 | Erros: 4 | Pesos: [0.4  0.75] | Vies: -0.6
Epoca  9 | Erros: 4 | Pesos: [0.35 0.8 ] | Vies: -0.8
Epoca 10 | Erros: 4 | Pesos: [0.35 0.7 ] | Vies: -1.0
Epoca 11 | Erros: 2 | Pesos: [-0.   0.4] | Vies: -1.2
Epoca 12 | Erros: 0 | Pesos: [-0.   0.4] | Vies: -1.2
-> Convergencia atingida na epoca 12.

=== Parâmetros Calibrados ===
Pesos finais (w1, w2): [-5.55111512e-16  4.00000000e-01]
Viés final (b): -1.2

=== Diagnóstico das Novas Transações ===
Transação A [np.float64(5.0), np.float64(4.0)] -> Classe: 1 -> SUSPEITA (Risco de Fraude)
Transação B [np.float64(7.0), np.float64(6.5)] -> Classe: 1 -> SUSPEITA (Ri

---
## Desafio 2: Predição de Risco de Churn com Classificador KNN

**Contexto:** antecipar risco de cancelamento de clientes SaaS com base em:
- `x1`: Dias de Inatividade (últimos 30 dias)
- `x2`: Chamados Críticos em Aberto no mês

O KNN não tem fase de treinamento com ajuste de pesos — a classificação é feita por **votação majoritária dos k vizinhos mais próximos**, calculados de forma vetorizada.

### 2.1 Funções do KNN
Cálculo vetorizado de distâncias (Euclidiana e Manhattan) e classificação por votação.

In [5]:
def calcular_distancias(X, ponto, metrica="euclidiana"):
    """Calcula distâncias vetorizadas entre 'ponto' e todas as linhas de X."""
    if metrica == "euclidiana":
        return np.linalg.norm(X - ponto, ord=2, axis=1)
    elif metrica == "manhattan":
        return np.linalg.norm(X - ponto, ord=1, axis=1)
    else:
        raise ValueError("Métrica inválida. Use \'euclidiana\' ou \'manhattan\'.")


def knn_classificar(X, y, ponto, k=3, metrica="euclidiana"):
    """
    Classifica 'ponto' por votação majoritária dos k vizinhos mais próximos.
    Retorna: classe predita, índices dos vizinhos, distâncias dos vizinhos.
    """
    distancias = calcular_distancias(X, ponto, metrica=metrica)
    vizinhos_idx = np.argsort(distancias)[:k]
    votos = y[vizinhos_idx]
    classe_predita = int(np.bincount(votos).argmax())
    return classe_predita, vizinhos_idx, distancias[vizinhos_idx]

### 2.2 Dados de Treino e Clientes sob Avaliação

In [6]:
X_treino_churn = np.array([
    [2.0, 1.0],
    [3.0, 0.0],
    [5.0, 1.0],
    [6.0, 2.0],
    [15.0, 4.0],
    [18.0, 5.0],
    [20.0, 4.0],
    [22.0, 6.0]
])

y_treino_churn = np.array([0, 0, 0, 0, 1, 1, 1, 1])

rotulo_classe = {0: "Baixo Risco", 1: "Alto Risco"}

clientes_teste = {
    "Cliente 1": {"ponto": np.array([13.0, 10.0]), "k": 5},
    "Cliente 2": {"ponto": np.array([7.0, 1.5]),  "k": 3},
}

### 2.3 Execução e Diagnóstico

In [7]:
for nome_cliente, info in clientes_teste.items():
    ponto = info["ponto"]
    k = info["k"]

    print(f"--- {nome_cliente} {list(ponto)} (K={k}) ---")

    for metrica in ["euclidiana", "manhattan"]:
        classe, idx_vizinhos, dists = knn_classificar(
            X_treino_churn, y_treino_churn, ponto, k=k, metrica=metrica
        )

        print(f"  Métrica: {metrica.capitalize()}")
        print(f"    Classe predita:        {rotulo_classe[classe]} (Classe {classe})")
        print(f"    Índices dos vizinhos:  {list(idx_vizinhos)}")
        print(f"    Distâncias calculadas: {np.round(dists, 4)}")

    print()

--- Cliente 1 [np.float64(13.0), np.float64(10.0)] (K=5) ---
  Métrica: Euclidiana
    Classe predita:        Alto Risco (Classe 1)
    Índices dos vizinhos:  [np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(3)]
    Distâncias calculadas: [ 6.3246  7.0711  9.2195  9.8489 10.6301]
  Métrica: Manhattan
    Classe predita:        Alto Risco (Classe 1)
    Índices dos vizinhos:  [np.int64(4), np.int64(5), np.int64(7), np.int64(6), np.int64(3)]
    Distâncias calculadas: [ 8. 10. 13. 13. 15.]

--- Cliente 2 [np.float64(7.0), np.float64(1.5)] (K=3) ---
  Métrica: Euclidiana
    Classe predita:        Baixo Risco (Classe 0)
    Índices dos vizinhos:  [np.int64(3), np.int64(2), np.int64(1)]
    Distâncias calculadas: [1.118  2.0616 4.272 ]
  Métrica: Manhattan
    Classe predita:        Baixo Risco (Classe 0)
    Índices dos vizinhos:  [np.int64(3), np.int64(2), np.int64(1)]
    Distâncias calculadas: [1.5 2.5 5.5]



---
## Desafio 3: Recomendação de Servidores Cloud por Similaridade Espacial

**Contexto:** recomendar máquinas virtuais (VMs) do catálogo mais similares, geometricamente, a um perfil de hardware demandado (vCPUs, RAM, SSD). Não há rótulos de classe — o objetivo é apenas encontrar as instâncias com menor distância euclidiana.

### 3.1 Função de Recomendação

In [8]:
def recomendar_servidores(catalogo, perfil_demandado, k=2, metrica="euclidiana"):
    """
    Recomenda os k servidores do catálogo mais próximos (geometricamente)
    do perfil de hardware demandado.
    Retorna: índices ordenados e distâncias correspondentes.
    """
    if metrica == "euclidiana":
        distancias = np.linalg.norm(catalogo - perfil_demandado, ord=2, axis=1)
    else:
        raise ValueError("Métrica inválida. Use \'euclidiana\'.")

    indices_ordenados = np.argsort(distancias)[:k]
    return indices_ordenados, distancias[indices_ordenados]

### 3.2 Catálogo de Instâncias e Perfil Demandado

In [9]:
catalogo_servidores = np.array([
    [2.0, 4.0, 50.0],
    [4.0, 8.0, 100.0],
    [8.0, 16.0, 250.0],
    [16.0, 32.0, 500.0],
    [32.0, 64.0, 1000.0],
    [64.0, 128.0, 2000.0]
])

nomes_servidores = [
    "Micro Instância Web",
    "Standard App Server",
    "Medium Backend & Cache",
    "Database Enterprise",
    "High Performance Computing",
    "Big Data & AI Training"
]

perfil_demandado = np.array([12.0, 28.0, 850.0])
k = 2

### 3.3 Execução e Ranking de Recomendações

In [10]:
print(f"Perfil de Hardware Demandado: {list(perfil_demandado)} (vCPUs, RAM GB, SSD GB)\n")

indices, distancias = recomendar_servidores(
    catalogo_servidores, perfil_demandado, k=k, metrica="euclidiana"
)

print(f"=== Ranking de Recomendações (Top {k}) ===")
for posicao, (idx, dist) in enumerate(zip(indices, distancias), start=1):
    nome = nomes_servidores[idx]
    vcpu, ram, ssd = catalogo_servidores[idx]
    print(f"{posicao}º lugar: {nome}")
    print(f"    Especificações: {vcpu:.0f} vCPUs | {ram:.0f} GB RAM | {ssd:.0f} GB SSD")
    print(f"    Distância até a demanda: {dist:.4f}\n")

Perfil de Hardware Demandado: [np.float64(12.0), np.float64(28.0), np.float64(850.0)] (vCPUs, RAM GB, SSD GB)

=== Ranking de Recomendações (Top 2) ===
1º lugar: High Performance Computing
    Especificações: 32 vCPUs | 64 GB RAM | 1000 GB SSD
    Distância até a demanda: 155.5506

2º lugar: Database Enterprise
    Especificações: 16 vCPUs | 32 GB RAM | 500 GB SSD
    Distância até a demanda: 350.0457

